# Notebook 02 — Preprocessing Pipeline Demo
## AI-Based Multilingual Cyberbullying Detection

This notebook walks through every stage of the preprocessing pipeline, showing
raw → cleaned → tokenized transformations for Roman Urdu social media text.

**Contents:**
1. TextPreprocessor Walkthrough
2. Raw vs Cleaned Text Examples
3. Roman Urdu Normalization Examples
4. Tokenization Output Visualization (m-BERT)
5. Token Length Analysis on Full Dataset
6. Truncation Impact Assessment
7. Preprocessing Summary

## 0. Imports & Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make project root importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.preprocessing import TextPreprocessor
from src import config

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

DATA_CSV = os.path.join(PROJECT_ROOT, 'data', 'train.csv')
df = pd.read_csv(DATA_CSV).dropna(subset=['text_content']).reset_index(drop=True)
print(f'Loaded {len(df):,} rows (1 null row dropped)')
print(f'MAX_LEN (tokenizer truncation): {config.MAX_LEN}')

## 1. TextPreprocessor Overview

The `TextPreprocessor` class (in `src/data/preprocessing.py`) runs a three-stage pipeline:

```
Raw text
   │
   ├─ clean_text()
   │     • Strip URLs          (https://..., www....)
   │     • Strip HTML tags     (<br>, <b>, etc.)
   │     • Strip @mentions     (@username)
   │     • Replace emojis/special chars with spaces
   │     • Normalize whitespace
   │
   └─ normalize_roman_urdu()   (only when lang == 'roman_urdu')
         • Lowercase
         • Ambiguous spelling normalization
           (aa→a, ee→i, oo→u)
```

In [ ]:
# Inspect the preprocessor source
import inspect
print(inspect.getsource(TextPreprocessor))

## 2. Raw vs Cleaned Text Examples

In [ ]:
preprocessor = TextPreprocessor()

# Synthetic examples that demonstrate each cleaning rule clearly
synthetic_examples = [
    {
        'scenario': 'URL removal',
        'raw': 'Check this out https://bit.ly/bully123 it is disgusting',
        'lang': 'en'
    },
    {
        'scenario': 'HTML tag removal',
        'raw': 'You are <b>so stupid</b> and <i>pathetic</i>',
        'lang': 'en'
    },
    {
        'scenario': '@mention removal',
        'raw': '@john_doe you are the worst person ever @admin ban this guy',
        'lang': 'en'
    },
    {
        'scenario': 'Emoji / special char removal',
        'raw': 'tum burey insaan ho 😡🔥 bilkul ghalat hai ye!',
        'lang': 'roman_urdu'
    },
    {
        'scenario': 'Multiple noise sources',
        'raw': '@user123 check www.hate.com <br> tum bahut bure ho!!!',
        'lang': 'roman_urdu'
    },
]

rows = []
for ex in synthetic_examples:
    cleaned = preprocessor.preprocess(ex['raw'], lang=ex['lang'])
    rows.append({'Scenario': ex['scenario'], 'Raw Text': ex['raw'],
                 'Cleaned Text': cleaned, 'Language': ex['lang']})

ex_df = pd.DataFrame(rows).set_index('Scenario')
pd.set_option('display.max_colwidth', 80)
display(ex_df)

In [ ]:
# Apply to actual dataset samples and show side-by-side
sample = df.sample(10, random_state=7).reset_index(drop=True)

sample['cleaned_text'] = sample.apply(
    lambda r: preprocessor.preprocess(str(r['text_content']), lang=r['language']), axis=1
)
sample['char_change'] = sample['cleaned_text'].str.len() - sample['text_content'].str.len()

comparison = sample[['text_content', 'cleaned_text', 'char_change', 'aggression']].copy()
comparison.columns = ['Raw', 'Cleaned', 'Char Δ', 'Aggression']
display(comparison)

## 3. Roman Urdu Normalization Examples

The dataset is 100% Roman Urdu — informal transliterated Urdu written in Latin script.
This creates unique challenges: highly variable spellings for the same word.

The normalizer applies:
- Lowercase conversion
- Vowel contraction: `aa→a`, `ee→i`, `oo→u`

In [ ]:
# Show how common Roman Urdu words are normalized
roman_examples = [
    ('Bohut', 'very/much — common spelling variant'),
    ('boohut', 'very/much — elongated'),
    ('Khaana', 'food — long vowel'),
    ('Neend', 'sleep — ee vowel'),
    ('Toofaan', 'storm — oo vowel'),
    ('BAAR BAAR', 'again and again — uppercase + long vowel'),
    ('Poora', 'complete/all'),
    ('ACHOO', 'good — uppercase + oo'),
    ('tum bahut burey ho', 'you are very bad — full phrase'),
    ('MAT KARO', 'don\'t do it — uppercase'),
]

norm_rows = []
for raw, meaning in roman_examples:
    normalized = preprocessor.normalize_roman_urdu(raw)
    norm_rows.append({'Raw (Roman Urdu)': raw, 'Normalized': normalized,
                      'Meaning': meaning,
                      'Changed': '✅' if raw.lower() != normalized else '—'})

norm_df = pd.DataFrame(norm_rows).set_index('Raw (Roman Urdu)')
display(norm_df)

In [ ]:
# How much does cleaning change text length across the full dataset?
df['cleaned'] = df.apply(
    lambda r: preprocessor.preprocess(str(r['text_content']), lang=r['language']), axis=1
)
df['raw_len']     = df['text_content'].str.len()
df['cleaned_len'] = df['cleaned'].str.len()
df['len_reduction'] = df['raw_len'] - df['cleaned_len']

print('Character reduction after cleaning:')
display(df['len_reduction'].describe().rename('Char Reduction').to_frame())

no_change = (df['len_reduction'] == 0).sum()
print(f'\nRows with NO change after cleaning: {no_change:,} ({no_change/len(df)*100:.1f}%)')
print('This is expected — Roman Urdu text here is mostly clean (no URLs or HTML).')

## 4. Tokenization Output Visualization (m-BERT)

The text encoder uses the m-BERT tokenizer (`bert-base-multilingual-cased`).
Understanding tokenization output is critical:
- `[CLS]` and `[SEP]` tokens are added automatically
- WordPiece splits unknown words into sub-word pieces (e.g., `playing → play ##ing`)
- Sequences longer than `MAX_LEN=128` are **truncated**

In [ ]:
# Try to import transformers; gracefully degrade if not installed
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL)
    TOKENIZER_AVAILABLE = True
    print(f'Tokenizer loaded: {config.TEXT_MODEL}')
    print(f'Vocab size: {tokenizer.vocab_size:,}')
except Exception as e:
    TOKENIZER_AVAILABLE = False
    print(f'Tokenizer not available (install transformers): {e}')
    print('Sections 4–6 will be skipped. Run: pip install transformers')

In [ ]:
if TOKENIZER_AVAILABLE:
    # Show tokenization for a few examples
    demo_texts = [
        ('tum bohut bure insaan ho', 'roman_urdu', 'Short aggressive post'),
        ('Shukar hay Australia jeeta:)', 'roman_urdu', 'Short positive — emoji context'),
        ('ahan Australia ne really boht acha khela yyaar they desrves but newzealand ka frst time tha to un k liye feeling sad ..', 'roman_urdu', 'Long neutral post'),
    ]
    
    for raw, lang, desc in demo_texts:
        cleaned = preprocessor.preprocess(raw, lang=lang)
        tokens = tokenizer.tokenize(cleaned)
        encoded = tokenizer.encode_plus(
            cleaned, max_length=config.MAX_LEN, padding='max_length',
            truncation=True, add_special_tokens=True
        )
        non_pad = sum(1 for x in encoded['attention_mask'] if x == 1)
        
        print(f'--- {desc} ---')
        print(f'  Raw       : {raw}')
        print(f'  Cleaned   : {cleaned}')
        print(f'  Tokens ({len(tokens)}): {tokens}')
        print(f'  Seq length: {non_pad} / {config.MAX_LEN} (incl. [CLS] + [SEP])')
        print(f'  Input IDs : {encoded["input_ids"][:non_pad]}')
        print()

In [ ]:
if TOKENIZER_AVAILABLE:
    # Visualize token sequence as color-coded blocks
    raw_text = 'tum log mujhe itna kyun tang karte ho main kuch nahi kiya aapka'
    cleaned  = preprocessor.preprocess(raw_text, lang='roman_urdu')
    tokens   = ['[CLS]'] + tokenizer.tokenize(cleaned) + ['[SEP]']
    
    fig, ax = plt.subplots(figsize=(14, 1.8))
    ax.set_xlim(0, len(tokens))
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    for i, tok in enumerate(tokens):
        if tok in ('[CLS]', '[SEP]'):
            color, alpha = '#e8c170', 1.0      # gold for special tokens
        elif tok.startswith('##'):
            color, alpha = '#a8d8ea', 1.0      # blue for subword continuations
        else:
            color, alpha = '#b8e0b8', 1.0      # green for whole words
        
        rect = mpatches.FancyBboxPatch((i + 0.05, 0.15), 0.85, 0.7,
                                        boxstyle='round,pad=0.05',
                                        facecolor=color, edgecolor='white', linewidth=1.2)
        ax.add_patch(rect)
        ax.text(i + 0.5, 0.5, tok, ha='center', va='center',
                fontsize=7.5, fontweight='bold' if tok in ('[CLS]','[SEP]') else 'normal')
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#e8c170', label='Special token ([CLS]/[SEP])'),
        mpatches.Patch(color='#b8e0b8', label='Whole word token'),
        mpatches.Patch(color='#a8d8ea', label='Subword continuation (##)'),
    ]
    ax.legend(handles=legend_patches, loc='upper right',
              bbox_to_anchor=(1.0, -0.05), ncol=3, fontsize=9)
    
    ax.set_title(f'Tokenization of: "{cleaned}" ({len(tokens)} tokens)',
                 fontweight='bold', pad=8)
    plt.tight_layout()
    plt.savefig('tokenization_visualization.png', bbox_inches='tight')
    plt.show()

## 5. Token Length Analysis on Full Dataset

In [ ]:
if TOKENIZER_AVAILABLE:
    print('Tokenizing full dataset (may take 1-2 minutes)...')
    token_lengths = []
    for _, row in df.iterrows():
        cleaned = preprocessor.preprocess(str(row['text_content']), lang=row['language'])
        toks = tokenizer.tokenize(cleaned)
        token_lengths.append(len(toks) + 2)  # +2 for [CLS] and [SEP]
    
    df['token_len'] = token_lengths
    print('Done!')
    print(df['token_len'].describe().rename('Token Length').to_frame())
else:
    print('Skipped — tokenizer not available.')

In [ ]:
if TOKENIZER_AVAILABLE and 'token_len' in df.columns:
    MAX_LEN = config.MAX_LEN
    truncated = (df['token_len'] > MAX_LEN).sum()
    truncated_pct = truncated / len(df) * 100
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df['token_len'], bins=40, color='#4C72B0', edgecolor='white', linewidth=0.5)
    ax.axvline(MAX_LEN, color='#C44E52', linestyle='--', linewidth=2.0,
               label=f'MAX_LEN = {MAX_LEN} (truncation boundary)')
    ax.axvline(df['token_len'].median(), color='#DD8452', linestyle='-', linewidth=1.8,
               label=f'Median = {df["token_len"].median():.0f}')
    ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 200],
                     MAX_LEN, df['token_len'].max(),
                     alpha=0.15, color='#C44E52', label=f'Truncated: {truncated:,} ({truncated_pct:.1f}%)')
    ax.set_xlabel('Token Length (incl. [CLS] and [SEP])')
    ax.set_ylabel('Frequency')
    ax.set_title('Token Length Distribution across Dataset', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('token_length_distribution.png', bbox_inches='tight')
    plt.show()
    
    print(f'Sequences exceeding MAX_LEN={MAX_LEN}: {truncated:,} ({truncated_pct:.1f}%)')
    if truncated_pct < 5:
        print('✅ Truncation impact is minimal — less than 5% of samples will be cut.')
    else:
        print(f'⚠️  {truncated_pct:.1f}% of samples are truncated. Consider increasing MAX_LEN.')

## 6. Truncation Impact Assessment

In [ ]:
if TOKENIZER_AVAILABLE and 'token_len' in df.columns:
    # Show the longest sequences and what gets cut off
    long_samples = df.nlargest(5, 'token_len')[['text_content', 'token_len', 'aggression']].copy()
    long_samples.columns = ['Text', 'Token Length', 'Aggression']
    long_samples['Tokens over limit'] = (long_samples['Token Length'] - config.MAX_LEN).clip(lower=0)
    print('Top 5 longest sequences (most truncation):')
    display(long_samples)
else:
    # Show character-based estimation without tokenizer
    long_samples = df.nlargest(5, 'char_len')[['text_content', 'char_len', 'aggression']]
    display(long_samples)

## 7. Preprocessing Summary

| Stage | Operation | Effect on this Dataset |
|---|---|---|
| **URL removal** | Strips `https://...` and `www....` | Minimal — no URLs found in Roman Urdu data |
| **HTML removal** | Strips `<tag>` patterns | Minimal — plain text social media |
| **@mention removal** | Strips `@username` | Occasional usernames stripped |
| **Emoji removal** | Non-word chars → space | Removes emoticons (`:p`, `:)`) — these carry sentiment! |
| **Whitespace normalization** | Collapse multiple spaces | Minor cleanup |
| **Lowercase** | Full lowercase (Roman Urdu only) | Major normalization — all caps posts normalized |
| **Vowel contraction** | `aa→a`, `ee→i`, `oo→u` | Reduces variant spellings |

### Known Limitation — Emoticons

The emoji regex `[^\w\s,.]` strips emoticons like `:p`, `:)`, `:D`.
In Roman Urdu social media, these carry significant sentiment information.
A potential improvement is to replace them with sentiment words before removal:

```python
# Example improvement (not currently in codebase)
EMOTICON_MAP = {':)': 'happy', ':(': 'sad', ':p': 'playful', ':D': 'laughing'}
for emote, word in EMOTICON_MAP.items():
    text = text.replace(emote, word)
```

### m-BERT Tokenization on Roman Urdu

m-BERT was trained on 104 languages but **not** on Roman Urdu specifically.
Roman Urdu words are split into many sub-word pieces, which:
- Increases effective token count per post
- May exceed `MAX_LEN` more than English text would
- Means MuRIL (which includes Urdu/Hindi) may handle this better

This is one motivation for comparing m-BERT vs MuRIL in `compare_models.py`.

In [ ]:
# Quick sanity check: run the full preprocessing pipeline on a batch
batch = df.sample(20, random_state=99)
batch['cleaned'] = batch.apply(
    lambda r: preprocessor.preprocess(str(r['text_content']), lang=r['language']), axis=1
)

empty_after_clean = (batch['cleaned'].str.strip() == '').sum()
print(f'Samples that become empty after preprocessing: {empty_after_clean}')
print(f'Preprocessing pipeline runs successfully on batch of {len(batch)} samples. ✅')